# 05 — Inference Demo

Demonstração educacional, local e reexecutável de inferência. O fluxo valida os artefatos finais antes da desserialização, usa somente inputs sintéticos independentes em memória e preserva `operational_prediction_available = false`.

## 1. Educational Inference Context

Esta etapa demonstra o uso do pipeline final apenas para estudo. O threshold foi selecionado na partição de validação, não constitui política operacional e nenhuma decisão sobre clientes é produzida.

## 2. Independent Project and Runtime Loading

In [1]:
from pathlib import Path
import sys
import pandas as pd


def discover_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "scripts" / "smoke_predict.py").is_file():
            return candidate
    raise RuntimeError("Project root could not be discovered from the current working directory.")


PROJECT_ROOT = discover_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.finalize_model import (
    load_and_validate_final_model_handoff,
    load_and_validate_inference_bundle,
)
from scripts.smoke_predict import (
    current_runtime_versions,
    load_validated_inference_pipeline,
    predict_educational,
    predict_educational_batch,
    validate_bundle_handoff_alignment,
    validate_inference_readiness,
    validate_loaded_pipeline_contract,
    validate_model_artifact_before_load,
    validate_runtime_compatibility,
)

HANDOFF_PATH = Path("artifacts/models/telco-customer-churn/final-model-handoff.json")
BUNDLE_PATH = Path("artifacts/models/telco-customer-churn/inference-bundle.json")


def relative_text(path: Path) -> str:
    return path.resolve().relative_to(PROJECT_ROOT).as_posix()


print({"project_root": ".", "handoff": HANDOFF_PATH.as_posix(), "bundle": BUNDLE_PATH.as_posix()})

{'project_root': '.', 'handoff': 'artifacts/models/telco-customer-churn/final-model-handoff.json', 'bundle': 'artifacts/models/telco-customer-churn/inference-bundle.json'}


## 3. Final-Model Handoff Integrity Gate

In [2]:
handoff = load_and_validate_final_model_handoff(
    project_root=PROJECT_ROOT,
    handoff_path=HANDOFF_PATH,
)
print({
    "dataset_slug": handoff["dataset_slug"],
    "selected_model_id": handoff["selected_model_id"],
    "final_model_handoff_ready": handoff["final_model_handoff_ready"],
    "educational_inference_demo_ready": handoff["educational_inference_demo_ready"],
})

{'dataset_slug': 'telco-customer-churn', 'selected_model_id': 'hist_gradient_boosting', 'final_model_handoff_ready': True, 'educational_inference_demo_ready': True}


## 4. Inference-Bundle Integrity Gate

In [3]:
bundle = load_and_validate_inference_bundle(
    project_root=PROJECT_ROOT,
    bundle_path=BUNDLE_PATH,
)
validate_inference_readiness(handoff, bundle)
validate_bundle_handoff_alignment(handoff, bundle)
print({
    "bundle_version": bundle["bundle_version"],
    "model_id": bundle["model_id"],
    "operational_modeling_ready": bundle["readiness"]["operational_modeling_ready"],
    "operational_prediction_available": bundle["output_contract"]["operational_prediction_available"],
})

{'bundle_version': '1.0.0', 'model_id': 'hist_gradient_boosting', 'operational_modeling_ready': False, 'operational_prediction_available': False}


## 5. Trusted Model Artifact and SHA-256 Gate

In [4]:
validated_model_path = validate_model_artifact_before_load(
    project_root=PROJECT_ROOT,
    bundle=bundle,
    handoff=handoff,
)
print({
    "model_artifact_path": relative_text(validated_model_path),
    "model_artifact_sha256": bundle["model_artifact_sha256"],
    "trusted_origin_confirmed_for_demo": True,
})

{'model_artifact_path': 'artifacts/models/telco-customer-churn/final-pipeline.joblib', 'model_artifact_sha256': '48da4c7aa56d5d08090e808f551f555740d18e791af39c4c3f373467ec296b8c', 'trusted_origin_confirmed_for_demo': True}


## 6. Runtime Compatibility Report

In [5]:
observed_runtime = current_runtime_versions()
expected_runtime = bundle["runtime_version_requirements"]
exact_runtime_report = validate_runtime_compatibility(
    expected_runtime,
    observed_versions=observed_runtime,
    mode="exact",
)
load_safe_runtime_report = validate_runtime_compatibility(
    expected_runtime,
    observed_versions=observed_runtime,
    mode="load_safe",
    raise_on_incompatible=False,
)
print({
    "expected": expected_runtime,
    "observed": observed_runtime,
    "exact": exact_runtime_report.as_dict(),
    "load_safe": load_safe_runtime_report.as_dict(),
})

{'expected': {'joblib': '1.5.3', 'pandas': '3.0.5', 'python': '3.13.13', 'scikit_learn': '1.9.0'}, 'observed': {'python': '3.13.13', 'pandas': '3.0.5', 'scikit_learn': '1.9.0', 'joblib': '1.5.3'}, 'exact': {'mode': 'exact', 'compatible': True, 'components': [{'component': 'python', 'expected': '3.13.13', 'observed': '3.13.13', 'compatible': True, 'status': 'compatible', 'detail': 'exact match'}, {'component': 'pandas', 'expected': '3.0.5', 'observed': '3.0.5', 'compatible': True, 'status': 'compatible', 'detail': 'exact match'}, {'component': 'scikit_learn', 'expected': '1.9.0', 'observed': '1.9.0', 'compatible': True, 'status': 'compatible', 'detail': 'exact match'}, {'component': 'joblib', 'expected': '1.5.3', 'observed': '1.5.3', 'compatible': True, 'status': 'compatible', 'detail': 'exact match'}], 'warnings': []}, 'load_safe': {'mode': 'load_safe', 'compatible': True, 'components': [{'component': 'python', 'expected': '3.13.13', 'observed': '3.13.13', 'compatible': True, 'status':

## 7. Trusted Pipeline Loading

In [6]:
pipeline, loaded_handoff, loaded_bundle, runtime_report = load_validated_inference_pipeline(
    project_root=PROJECT_ROOT,
    handoff_path=HANDOFF_PATH,
    bundle_path=BUNDLE_PATH,
    trusted_source=True,
)
print({
    "loaded": True,
    "runtime_load_safe": runtime_report.compatible,
    "model_state_fingerprint": loaded_bundle["model_state_fingerprint"],
})

{'loaded': True, 'runtime_load_safe': True, 'model_state_fingerprint': 'bcce6491415e3a17d277858b6b71f04bc0ba14174b0a11e0dc0544812110c34f'}


## 8. Loaded Pipeline Contract Inspection

In [7]:
validate_loaded_pipeline_contract(pipeline, bundle=loaded_bundle)
preprocess = pipeline.named_steps["preprocess"]
model = pipeline.named_steps["model"]
print({
    "pipeline_class": f"{pipeline.__class__.__module__}.{pipeline.__class__.__name__}",
    "steps": list(pipeline.named_steps),
    "preprocess_class": preprocess.__class__.__name__,
    "model_class": model.__class__.__name__,
    "classes": model.classes_.tolist(),
    "transformed_feature_count": len(preprocess.get_feature_names_out()),
})

{'pipeline_class': 'sklearn.pipeline.Pipeline', 'steps': ['preprocess', 'model'], 'preprocess_class': 'ColumnTransformer', 'model_class': 'HistGradientBoostingClassifier', 'classes': [0, 1], 'transformed_feature_count': 46}


## 9. Input Schema and Feature Order

In [8]:
input_schema = pd.DataFrame(
    {
        "feature": loaded_bundle["feature_columns"],
        "expected_dtype": [
            loaded_bundle["expected_input_dtypes"][feature]
            for feature in loaded_bundle["feature_columns"]
        ],
        "role": [
            "categorical" if feature in loaded_bundle["categorical_features"] else "numerical"
            for feature in loaded_bundle["feature_columns"]
        ],
    }
)
input_schema

,feature,expected_dtype,role
0,gender,string,categorical
1,SeniorCitizen,integer,categorical
2,Partner,string,categorical
3,Dependents,string,categorical
4,tenure,integer,numerical
5,PhoneService,string,categorical
6,MultipleLines,string,categorical
7,InternetService,string,categorical
8,OnlineSecurity,string,categorical
9,OnlineBackup,string,categorical


## 10. Missing-Value Materialization Policy

In [9]:
print({
    "preparation_rules": loaded_bundle["missing_value_policy"]["preparation_rules"],
    "learned_imputation_in_final_pipeline": loaded_bundle["missing_value_policy"]["learned_imputation_in_final_pipeline"],
})

{'preparation_rules': [{'blank_replacement': 0.0, 'column': 'TotalCharges', 'condition_column': 'tenure', 'condition_value': 0, 'strip_strings': True}], 'learned_imputation_in_final_pipeline': False}


## 11. Unknown-Category Policy

In [10]:
print({
    "policy": loaded_bundle["unknown_category_policy"],
    "encoder_handle_unknown": preprocess.named_transformers_["categorical"].handle_unknown,
    "behavior": "preserve, aggregate report, then let the fitted encoder ignore",
})

{'policy': 'ignore_and_report', 'encoder_handle_unknown': 'ignore', 'behavior': 'preserve, aggregate report, then let the fitted encoder ignore'}


## 12. Independent Single-Row Inference

In [11]:
single_input = {
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "No",
    "tenure": 24,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "DSL",
    "OnlineSecurity": "Yes",
    "OnlineBackup": "No",
    "DeviceProtection": "Yes",
    "TechSupport": "No",
    "StreamingTV": "No",
    "StreamingMovies": "Yes",
    "Contract": "One year",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Credit card (automatic)",
    "MonthlyCharges": 65.40,
    "TotalCharges": 1569.60,
}
single_result = predict_educational(
    pipeline,
    single_input,
    bundle=loaded_bundle,
    runtime_report=runtime_report,
)
single_result

{'positive_class_probability': 0.10520392842280891,
 'educational_prediction_encoded': 0,
 'educational_prediction_label': 'No',
 'educational_threshold': 0.2577809673219062,
 'operational_prediction_available': False,
 'unknown_categories_report': {},
 'input_materializations_applied': {},
 'runtime_compatibility_confirmed': True,
 'runtime_compatibility_report': {'mode': 'load_safe',
  'compatible': True,
  'components': [{'component': 'python',
    'expected': '3.13.13',
    'observed': '3.13.13',
    'compatible': True,
    'status': 'compatible',
    'detail': 'exact match'},
   {'component': 'pandas',
    'expected': '3.0.5',
    'observed': '3.0.5',
    'compatible': True,
    'status': 'compatible',
    'detail': 'exact match'},
   {'component': 'scikit_learn',
    'expected': '1.9.0',
    'observed': '1.9.0',
    'compatible': True,
    'status': 'compatible',
    'detail': 'exact match'},
   {'component': 'joblib',
    'expected': '1.5.3',
    'observed': '1.5.3',
    'compat

## 13. Educational Threshold Interpretation

In [12]:
print({
    "educational_threshold": loaded_bundle["educational_decision_threshold"],
    "scenario": loaded_bundle["threshold_scenario"],
    "selection_partition": loaded_bundle["threshold_selection_partition"],
    "purpose": "educational",
    "operational_threshold": loaded_bundle["operational_threshold"],
    "operational_prediction_available": False,
})

{'educational_threshold': 0.2577809673219062, 'scenario': 'minimum_recall_0_80', 'selection_partition': 'validation', 'purpose': 'educational', 'operational_threshold': 'unresolved', 'operational_prediction_available': False}


## 14. Declared Missing-Value Example

In [13]:
missing_value_input = {
    **single_input,
    "tenure": 0,
    "TotalCharges": "   ",
    "Contract": "Month-to-month",
}
missing_value_result = predict_educational(
    pipeline,
    missing_value_input,
    bundle=loaded_bundle,
    runtime_report=runtime_report,
)
missing_value_result

{'positive_class_probability': 0.5464823173205838,
 'educational_prediction_encoded': 1,
 'educational_prediction_label': 'Yes',
 'educational_threshold': 0.2577809673219062,
 'operational_prediction_available': False,
 'unknown_categories_report': {},
 'input_materializations_applied': {'TotalCharges': 1},
 'runtime_compatibility_confirmed': True,
 'runtime_compatibility_report': {'mode': 'load_safe',
  'compatible': True,
  'components': [{'component': 'python',
    'expected': '3.13.13',
    'observed': '3.13.13',
    'compatible': True,
    'status': 'compatible',
    'detail': 'exact match'},
   {'component': 'pandas',
    'expected': '3.0.5',
    'observed': '3.0.5',
    'compatible': True,
    'status': 'compatible',
    'detail': 'exact match'},
   {'component': 'scikit_learn',
    'expected': '1.9.0',
    'observed': '1.9.0',
    'compatible': True,
    'status': 'compatible',
    'detail': 'exact match'},
   {'component': 'joblib',
    'expected': '1.5.3',
    'observed': '1.

## 15. Unknown-Category Example

In [14]:
unknown_category_input = {
    **single_input,
    "PaymentMethod": "Synthetic digital wallet",
    "tenure": 11,
    "MonthlyCharges": 72.25,
    "TotalCharges": 794.75,
}
unknown_category_result = predict_educational(
    pipeline,
    unknown_category_input,
    bundle=loaded_bundle,
    runtime_report=runtime_report,
)
unknown_category_result

{'positive_class_probability': 0.12868196698469359,
 'educational_prediction_encoded': 0,
 'educational_prediction_label': 'No',
 'educational_threshold': 0.2577809673219062,
 'operational_prediction_available': False,
 'unknown_categories_report': {'PaymentMethod': ['Synthetic digital wallet']},
 'input_materializations_applied': {},
 'runtime_compatibility_confirmed': True,
 'runtime_compatibility_report': {'mode': 'load_safe',
  'compatible': True,
  'components': [{'component': 'python',
    'expected': '3.13.13',
    'observed': '3.13.13',
    'compatible': True,
    'status': 'compatible',
    'detail': 'exact match'},
   {'component': 'pandas',
    'expected': '3.0.5',
    'observed': '3.0.5',
    'compatible': True,
    'status': 'compatible',
    'detail': 'exact match'},
   {'component': 'scikit_learn',
    'expected': '1.9.0',
    'observed': '1.9.0',
    'compatible': True,
    'status': 'compatible',
    'detail': 'exact match'},
   {'component': 'joblib',
    'expected': 

## 16. Independent Batch Inference

In [15]:
batch_input = pd.DataFrame(
    [
        {
            **single_input,
            "gender": "Male",
            "tenure": 8,
            "Contract": "Month-to-month",
            "MonthlyCharges": 84.10,
            "TotalCharges": 672.80,
        },
        {
            **single_input,
            "SeniorCitizen": 1,
            "Partner": "No",
            "tenure": 48,
            "InternetService": "Fiber optic",
            "MonthlyCharges": 99.90,
            "TotalCharges": 4795.20,
        },
        {
            **single_input,
            "Dependents": "Yes",
            "tenure": 3,
            "Contract": "Two year",
            "MonthlyCharges": 41.50,
            "TotalCharges": 124.50,
        },
    ],
    index=pd.Index(["synthetic-a", "synthetic-b", "synthetic-c"], name="case"),
)
batch_result = predict_educational_batch(
    pipeline,
    batch_input,
    bundle=loaded_bundle,
    runtime_report=runtime_report,
)
batch_result

,positive_class_probability,educational_prediction_encoded,educational_prediction_label,educational_threshold,operational_prediction_available,unknown_categories_report,input_materializations_applied,runtime_compatibility_confirmed,runtime_compatibility_report,model_state_fingerprint
case,,,,,,,,,,
synthetic-a,0.252437,0,No,0.257781,False,{},{},True,"{'mode': 'load_safe', 'compatible': True, 'com...",bcce6491415e3a17d277858b6b71f04bc0ba14174b0a11...
synthetic-b,0.15602,0,No,0.257781,False,{},{},True,"{'mode': 'load_safe', 'compatible': True, 'com...",bcce6491415e3a17d277858b6b71f04bc0ba14174b0a11...
synthetic-c,0.056639,0,No,0.257781,False,{},{},True,"{'mode': 'load_safe', 'compatible': True, 'com...",bcce6491415e3a17d277858b6b71f04bc0ba14174b0a11...


## 17. Output Contract

In [16]:
required_output_fields = {
    "positive_class_probability",
    "educational_prediction_encoded",
    "educational_prediction_label",
    "educational_threshold",
    "operational_prediction_available",
}
assert required_output_fields.issubset(batch_result.columns)
assert batch_result["operational_prediction_available"].eq(False).all()
print({
    "rows": len(batch_result),
    "required_fields_present": True,
    "operational_prediction_available": False,
    "persisted": False,
})

{'rows': 3, 'required_fields_present': True, 'operational_prediction_available': False, 'persisted': False}


## 18. Smoke Validation and Independent Reload

In [17]:
reloaded_pipeline, reloaded_handoff, reloaded_bundle, reloaded_runtime_report = load_validated_inference_pipeline(
    project_root=PROJECT_ROOT,
    handoff_path=HANDOFF_PATH,
    bundle_path=BUNDLE_PATH,
    trusted_source=True,
)
validate_loaded_pipeline_contract(reloaded_pipeline, bundle=reloaded_bundle)
reload_smoke_result = predict_educational(
    reloaded_pipeline,
    {
        **single_input,
        "tenure": 36,
        "Contract": "Two year",
        "MonthlyCharges": 55.75,
        "TotalCharges": 2007.00,
    },
    bundle=reloaded_bundle,
    runtime_report=reloaded_runtime_report,
)
print({
    "independent_reload_completed": True,
    "same_model_state_fingerprint": reloaded_bundle["model_state_fingerprint"] == loaded_bundle["model_state_fingerprint"],
    "operational_prediction_available": reload_smoke_result["operational_prediction_available"],
})

{'independent_reload_completed': True, 'same_model_state_fingerprint': True, 'operational_prediction_available': False}


## 19. Operational Limitations

- A demonstração é educacional e baseada em um snapshot.
- O threshold foi selecionado em validação e não é uma política operacional.
- Validade operacional, contrato temporal e disponibilidade de features permanecem não confirmados.
- Nenhum input, probabilidade ou resultado é persistido.
- Nenhum endpoint ou API é implementado nesta etapa.

## 20. Study Completion Handoff

In [18]:
study_completion = {
    "educational_inference_demo_completed": bool(
        runtime_report.compatible
        and reloaded_runtime_report.compatible
        and reload_smoke_result["operational_prediction_available"] is False
    ),
    "educational_final_model_completed": loaded_handoff["educational_final_model_completed"],
    "final_model_trained": loaded_handoff["final_model_trained"],
    "model_artifact_materialized": loaded_handoff["model_artifact_materialized"],
    "model_bundle_materialized": loaded_handoff["model_bundle_materialized"],
    "final_model_handoff_ready": loaded_handoff["final_model_handoff_ready"],
    "educational_inference_demo_ready": loaded_handoff["educational_inference_demo_ready"],
    "operational_modeling_ready": loaded_handoff["operational_modeling_ready"],
    "operational_validity": loaded_handoff["operational_validity"],
    "operational_threshold": loaded_handoff["operational_threshold"],
    "temporal_contract_status": loaded_handoff["temporal_contract_status"],
    "feature_inference_availability": loaded_handoff["feature_inference_availability"],
    "api_implemented": loaded_handoff["api_implemented"],
    "operational_prediction_available": False,
}
study_completion

{'educational_inference_demo_completed': True,
 'educational_final_model_completed': True,
 'final_model_trained': True,
 'model_artifact_materialized': True,
 'model_bundle_materialized': True,
 'final_model_handoff_ready': True,
 'educational_inference_demo_ready': True,
 'operational_modeling_ready': False,
 'operational_validity': 'unconfirmed',
 'operational_threshold': 'unresolved',
 'temporal_contract_status': 'unresolved',
 'feature_inference_availability': 'unconfirmed',
 'api_implemented': False,
 'operational_prediction_available': False}